# Embed the medxir corpus with Octen/Octen-Embedding-8B

Generated by `scripts/build_embed_corpus_notebook.py`. Edit the generator, not this file.

Run on RTX Pro 6000 with Internet OFF. Attach `boinhbo/medxir-chunk-corpus` and
`boinhbo/medxir-package-octen8b`. The notebook stops at once on any other GPU.
A shard runs the parts whose index in the sorted list is `SHARD` modulo `SHARDS`.


In [ ]:
import glob
import json
import os
import time

# Parameters: change here only.
MODEL = 'Octen/Octen-Embedding-8B'
MODEL_DIR = 'Octen-Embedding-8B'
DOC_PREFIX = ' '         # text put before every chunk (Octen's document prompt is a single space)
SHARD, SHARDS = 2, 3  # this run embeds parts with index % SHARDS == SHARD
ONLY_PARTS = None          # part keys like 'vi-part-007' to embed (corpus v2 additions); None = every part of the shard
EXPECT_GPU = '6000'          # refuse to run on any other GPU
BATCH = 16              # 0.6B/4B: best batch in the RTX Pro 6000 benchmark; 8B: not benchmarked, 16 fits in 96 GB
MAX_LENGTH = 1024            # longest chunk is ~880 tokens, so nothing is truncated
TIME_BUDGET_S = 8 * 3600  # stop between parts before the session limit
OUT = '/kaggle/working'
os.makedirs(f'{OUT}/vectors', exist_ok=True)

## 1. GPU check (stops here on the wrong GPU)


In [ ]:
import subprocess

import torch

print(subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total,driver_version', '--format=csv'],
                     capture_output=True, text=True).stdout)
GPU_NAME = torch.cuda.get_device_name(0)
assert EXPECT_GPU in GPU_NAME, f'expected a GPU named *{EXPECT_GPU}*, got {GPU_NAME!r}; stop here'
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
print('GPU:', GPU_NAME, '| dtype:', DTYPE, '| torch', torch.__version__)

## 2. Model (offline, from the attached package)


In [ ]:
from transformers import AutoModel, AutoTokenizer

os.environ['HF_HUB_OFFLINE'] = '1'
hits = sorted(glob.glob(f'/kaggle/input/**/{MODEL_DIR}/config.json', recursive=True))
assert hits, f'attach boinhbo/medxir-package-models (kernel source) with {MODEL_DIR} before Run All'
SRC = os.path.dirname(hits[0])
start = time.perf_counter()
tok = AutoTokenizer.from_pretrained(SRC)
tok.padding_side = 'left'
model = AutoModel.from_pretrained(SRC, torch_dtype=DTYPE, attn_implementation='sdpa').cuda().eval()
LOAD_S = round(time.perf_counter() - start, 1)
print('loaded', SRC, 'in', LOAD_S, 's | hidden size', model.config.hidden_size)

## 3. Embed every chunk, part by part


In [ ]:
import numpy as np
import torch.nn.functional as F


def last_token_pool(hidden, mask):
    """Last-token pooling from the Qwen3-Embedding model card (handles left and right padding)."""
    if mask[:, -1].sum() == mask.shape[0]:
        return hidden[:, -1]
    lengths = mask.sum(dim=1) - 1
    return hidden[torch.arange(hidden.shape[0], device=hidden.device), lengths]


@torch.inference_mode()
def embed_texts(texts):
    """Float16 unit vectors in the input order, and the number of tokens processed."""
    ids = tok(texts, truncation=True, max_length=MAX_LENGTH)['input_ids']
    order = np.argsort([-len(x) for x in ids], kind='stable')  # longest first: less padding
    out = np.empty((len(texts), model.config.hidden_size), dtype=np.float16)
    for start in range(0, len(order), BATCH):
        index = order[start:start + BATCH]
        enc = tok.pad({'input_ids': [ids[i] for i in index]}, return_tensors='pt').to('cuda')
        hidden = model(input_ids=enc['input_ids'], attention_mask=enc['attention_mask'], use_cache=False).last_hidden_state
        vec = F.normalize(last_token_pool(hidden, enc['attention_mask']).float(), p=2, dim=1)
        out[index] = vec.cpu().numpy().astype(np.float16)
    return out, sum(len(x) for x in ids)

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

parts = sorted(glob.glob('/kaggle/input/**/chunks/*/part-*.parquet', recursive=True))
assert parts, 'attach boinhbo/medxir-chunk-corpus (kernel source) before Run All'
parts = [path for index, path in enumerate(parts) if index % SHARDS == SHARD]
if ONLY_PARTS is not None:
    parts = [path for path in parts
             if f"{path.split(os.sep)[-2]}-{os.path.basename(path).removesuffix('.parquet')}" in ONLY_PARTS]
    assert len(parts) == len(ONLY_PARTS), f'expected parts {ONLY_PARTS}, found {parts}'
print(len(parts), 'parts in shard', SHARD, 'of', SHARDS)
progress, started = [], time.perf_counter()
for path in parts:
    lang = path.split(os.sep)[-2]
    key = f"{lang}-{os.path.basename(path).removesuffix('.parquet')}"
    target = f'{OUT}/vectors/{key}.npy'
    if os.path.exists(target):
        print('skip', key, '(already written)')
        continue
    if time.perf_counter() - started > TIME_BUDGET_S:
        print('time budget spent; stopping before', key, '- the remaining parts are not written')
        break
    table = pq.read_table(path, columns=['chunk_id', 'chunk_text'])
    t0 = time.perf_counter()
    vectors, tokens = embed_texts([DOC_PREFIX + text for text in table.column('chunk_text').to_pylist()])
    torch.cuda.synchronize()
    seconds = time.perf_counter() - t0
    # Write under a temporary name, then rename, so an interrupted write never looks complete.
    np.save(f'{OUT}/vectors/{key}.tmp.npy', vectors)
    pq.write_table(table.select(['chunk_id']), f'{OUT}/vectors/{key}-ids.parquet')
    os.replace(f'{OUT}/vectors/{key}.tmp.npy', target)
    nan_rows = int((~np.isfinite(vectors.astype(np.float32))).any(axis=1).sum())
    progress.append(dict(part=key, chunks=table.num_rows, tokens=tokens, seconds=round(seconds, 1),
                         tokens_per_s=round(tokens / seconds), nan_rows=nan_rows))
    print(progress[-1])
    del table, vectors
    torch.cuda.empty_cache()

## 4. Manifest


In [ ]:
expected = sorted(os.path.basename(p).removesuffix('.parquet') for p in parts)
done = sorted(f.removesuffix('.npy') for f in os.listdir(f'{OUT}/vectors') if f.endswith('.npy'))
manifest = dict(
    model=MODEL, doc_prefix=DOC_PREFIX, model_dir=SRC, shard=SHARD, shards=SHARDS, gpu=GPU_NAME, dtype=str(DTYPE), batch=BATCH, max_length=MAX_LENGTH,
    dim=model.config.hidden_size, load_s=LOAD_S,
    complete=len(done) == len(expected), parts_done=len(done), parts_total=len(expected),
    chunks=sum(p['chunks'] for p in progress), tokens=sum(p['tokens'] for p in progress),
    nan_rows=sum(p['nan_rows'] for p in progress),
    run_seconds=round(time.perf_counter() - started, 1), parts=progress,
)
with open(f'{OUT}/embed_manifest.json', 'w', encoding='utf-8') as file:
    json.dump(manifest, file, ensure_ascii=False, indent=2)
print({k: v for k, v in manifest.items() if k != 'parts'})